In [ ]:
import os
import sys

import collections
import xarray as xr
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo

# settings
%config InlineBackend.figure_format = 'retina'

In [ ]:
### +++ LOAD CLIMATE DATA +++ ###

diags=['sst','nt_Heat_Atl','sf_Atl']
dat={ 'ctrl':{}, 'max.global':{} }

for key in dat.keys():    
    for varn in diags:
        dat[key][varn] = xr.open_dataset(f'aogcm.{key}.{varn}.nc')[varn]
        
## calculate annual mean fields for sst and streamfunction
# number of years to omit 
omit_yrs = 30
n_omit = omit_yrs * 12
tmax = 350 * 12
sst = {}
sf_Atl = {}
for key in dat.keys(): 
    sst[key] = dat[key]['sst'].isel(time=slice(n_omit,tmax)).mean(dim='time')
    sf_Atl[key] = dat[key]['sf_Atl'].isel(time=slice(n_omit,tmax)).mean(dim='time').where(ctrl > -6.8550344e+20, np.nan)
    
## Get latitude-weighted time-series of northward heat transport in the Atlantic
nt_Heat_Atl_monthly_weighted = {}
nt_Heat_Atl_ann_weighted = {}
lats = dat['ctrl']['nt_Heat_Atl'].lato2
weights=np.cos(np.deg2rad(lato2))
weights.name='weights'
for key in dat.keys():
    nt_Heat_Atl_monthly_weighted[key] = dat[key]['nt_Heat_Atl'].weighted(weights).mean(('lato2')) # apply weights
    nt_Heat_Atl_ann_weighted[key] = nt_Heat_Atl_monthly_weighted[key].groupby('time.year').mean(dim='time') # calculate annual means

In [ ]:
### +++ LOAD TOPO DATA +++ ###

# set file info
dpath0='../topo_files' # top level data directory
files={ 'ctrl':{}, 'max':{} }
files['ctrl'] = f'{dpath0}/e2pt1.ctrl.zsurf.nc'
files['max']  = f'{dpath0}/e2pt1.max.global.zsurf.nc'

# init dictionaries
topo={ 'ctrl':{}, 'max':{} }

for key in files.keys():
    topo[key] = xr.open_dataset(files[key]).zatmo

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# plot specs
legend_kw={'loc':0, 'fontsize':11, 'labelcolor':'linecolor', 'frameon':True}
text_kw={'weight':'bold', 'color':'k', 'ha':'left', 'va':'bottom'}
line_kw={'ls': '-', 'lw': 1}
tkw={'color': 'k', 'labelsize': 12, 'labelcolor': 'k'}
# timeseries tick info
tm=np.linspace(0.,len(nt_Heat_Atl_ann_weighted['ctrl'].year),len(nt_Heat_Atl_monthly_weighted['ctrl'].time))
ty=np.linspace(0.,len(nt_Heat_Atl_ann_weighted['ctrl'].year),len(nt_Heat_Atl_ann_weighted['ctrl'].year))
# map specs
lat=topo['ctrl'].lat
lon=topo['ctrl'].lon
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-150., -55., -55., 30.]
# sst colormap
cmap=cm.RdBu_r
vmin=-8
vmax=8
levels=np.linspace(vmin, vmax, 17)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
# sf colormap
cmap2 = cm.YlGnBu
vmin2 = -10
vmax2 = 30
levels2 = np.linspace(vmin2, vmax2, 21)
norm2 = mpl.colors.BoundaryNorm(levels2, cmap2.N)

# ------------------- #
#      Make Plot      #
# ------------------- #
fig=plt.figure(figsize=(10, 15), layout='constrained') 

## SST map
ax=plt.subplot2grid((3,2), (0, 0), colspan=2, rowspan=1, projection=proj) 
ax.text(-180,91,'a. MAX$_{global}$ $-$ CTRL $\Delta$SST', size=14, **text_kw)
cf=ax.pcolormesh(lon, lat, (sst['max']-sst['ctrl']),
                 cmap=cmap, norm=norm, transform=trans)
# add topo diff contours
zvar=topo['max']-topo['ctrl']
cs=ax.contour(lon, lat, zvar, levels=np.linspace(500, 4500, 9),
              linewidths=0.75, colors='k', transform=trans, zorder=7)
# settings
ax.coastlines(color='k')
ax.set_global() 
gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
gl.xformatter=LONGITUDE_FORMATTER; gl.yformatter=LATITUDE_FORMATTER
gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':10}
gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':10}
# add colorbar
cax=fig.add_axes([.975, 0.71, 0.0175, 0.25])
cbar=fig.colorbar(cf, orientation='vertical', extend='both', cax=cax)
cbar.set_label('[°C]', y=1.115, labelpad=-25, rotation=0, size=12, fontweight='bold', ha='center')
cbar.ax.tick_params(labelsize=12)
    
## Streamfunction
# ctrl
ax=plt.subplot2grid((3,2), (1, 0), colspan=1, rowspan=1) 
ax.text(-35,-75,'b. ATLANTIC STREAMFUNCTION ', size=14, **text_kw)
ax.text(-33.4,4935, 'CTRL', size=18, **text_kw,
           bbox=dict(facecolor='white', edgecolor='black'))
ax.contourf(lato, depth, sf_Atl['ctrl'], cmap=cmap2, levels=levels2, extend='both')
ax.set(xlabel="Latitude [°N]", xlim=[-35, 82],
       ylabel="Depth [mbsl]", ylim=[5000, 0])
# max
ax=plt.subplot2grid((3,2), (1, 1), colspan=1, rowspan=1) 
ax.text(-33.4,4935, 'MAX$_{global}$', size=18, **text_kw,
           bbox=dict(facecolor='white', edgecolor='black'))
cf2=ax.contourf(lato, depth, sf_Atl['max'], cmap=cmap2, levels=levels2, extend='both')
ax.set(xlabel="Latitude [°N]", xlim=[-35, 82],
       ylim=[5000, 0])
# add colorbar
cax=fig.add_axes([1.0, 0.375, 0.0175, 0.26])
cbar=fig.colorbar(cf2, orientation='vertical', extend='both', cax=cax)
cbar.set_label('[Sv]', y=1.115, labelpad=-25, rotation=0, size=12, fontweight='bold', ha='center')
cbar.ax.tick_params(labelsize=12)
    
    
## Heat Transport timeseries
ax=plt.subplot2grid((3,2), (2, 0), colspan=2, rowspan=1) 
ax.text(0,1,'c. ATLANTIC NORTHWARD HEAT TRANSPORT ', size=14, **text_kw)
# Control
ax.plot(tm, nt_Heat_Atl_monthly_weighted['ctrl'], c='grey', **line_kw, label='CTRL')
ax.plot(ty, nt_Heat_Atl_ann_weighted['ctrl'], c='k', **line_kw, label='_Hidden')
# Max
ax.plot(tm, nt_Heat_Atl_monthly_weighted['max'], c='r', **line_kw, label='MAX$_{global}$')
ax.plot(ty, nt_Heat_Atl_ann_weighted['max'], c='k', **line_kw, label='_Hidden')
# 0 line
ax.axhline(y=0, color='gray', lw=1, ls='--', label='_Hidden')
# settings
ax.set_xlabel('SIMULATION YEAR', size=12)
ax.set_ylabel('[10$^{15}$ W]', size=12)
ax.set(xlim=[0,350], ylim=[-.4,1])
ax.legend(**legend_kw)
ax.tick_params(**tkw)